<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2025</strong></center>

### Cuerpo Docente:

- Profesores: Stefano Schiappacasse, Sebastián Tinoco
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Angelo Muñoz, Valentina Zúñiga

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Samuel Vejar
- Nombre de alumno 2: David Valenzuela


### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/...../)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [1]:
!pip install -qq xgboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 13.4 MB/s eta 0:00:00


# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [2]:
import pandas as pd

# Cargar el dataset
df = pd.read_csv('sales.csv')

# Mostrar las primeras filas
print(df.head())


   id      date    city       lat      long     pop    shop        brand  \
0   0  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
1   1  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
2   2  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
3   3  31/01/12  Athens  37.97945  23.71622  672130  shop_1   adult-cola   
4   4  31/01/12  Athens  37.97945  23.71622  672130  shop_1   adult-cola   

  container capacity  price  quantity  
0     glass    500ml   0.96     13280  
1   plastic    1.5lt   2.86      6727  
2       can    330ml   0.87      9848  
3     glass    500ml   1.00     20050  
4       can    330ml   0.39     25696  


In [5]:
df.columns

Index(['id', 'date', 'city', 'lat', 'long', 'pop', 'shop', 'brand',
       'container', 'capacity', 'price', 'quantity'],
      dtype='object')

## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
#PASO1
# 1. Cargar y dividir el dataset

df = pd.read_csv('sales.csv')

# Variable objetivo
TARGET = 'quantity'

# División: 70% train, 20% validation, 10% test
df_train, df_temp = train_test_split(df, test_size=0.3, random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=1/3, random_state=42)


#PASO 2

#  FunctionTransformer para fecha

def extract_date_parts(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%y')
    df['day'] = df['date'].dt.day.astype('category')
    df['month'] = df['date'].dt.month.astype('category')
    df['year'] = df['date'].dt.year.astype('category')
    return df.drop(columns=['date'])

date_transformer = FunctionTransformer(extract_date_parts)

#PASO 3
# ColumnTransformer con escalamiento y OHE

numeric_features = ['lat', 'long', 'pop', 'price']
categorical_features = ['city', 'shop', 'brand', 'container', 'capacity', 'day', 'month', 'year']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
).set_output(transform='pandas')


#PASO 4
# Pipeline completo con DummyRegressor

pipeline_dummy = Pipeline(steps=[
    ('date', date_transformer),
    ('preprocess', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])

#PASO 5
# Entrenamiento y evaluación

X_train = df_train.drop(columns=['id', TARGET])
y_train = df_train[TARGET]

X_val = df_val.drop(columns=['id', TARGET])
y_val = df_val[TARGET]

# Entrenar el pipeline
pipeline_dummy.fit(X_train, y_train)

# Evaluar en validación
y_pred = pipeline_dummy.predict(X_val)
mae_dummy = mean_absolute_error(y_val, y_pred)

print(f"MAE usando DummyRegressor: {mae_dummy:.2f}")



MAE usando DummyRegressor: 13298.50


*RESPUESTA A PREGUNTA 5*
 la métrica Mean Absolute Error (MAE) nos indica cuántas unidades de venta en promedio nos equivocamos al hacer una predicción.

Un MAE alto significa que el modelo predice mal la demanda, lo cual puede llevar a: Sobreproducción (genera costos) o a una Subproducción (pérdida de ventas por falta de stock).

In [19]:
#PASO 6
from xgboost import XGBRegressor

# Pipeline con XGBRegressor (modelo real)
pipeline_xgb = Pipeline(steps=[
    ('date', date_transformer),
    ('preprocess', preprocessor),
    ('regressor', XGBRegressor(random_state=42))
])

# Entrenamiento
pipeline_xgb.fit(X_train, y_train)

# Predicción y evaluación
y_pred_xgb = pipeline_xgb.predict(X_val)
mae_xgb = mean_absolute_error(y_val, y_pred_xgb)

print(f"MAE usando XGBRegressor: {mae_xgb:.2f}")


MAE usando XGBRegressor: 2433.32


*RESPUESTA A PREGUNTA 6*

El MAE bajó de 13,298.50 a 2,433.32, lo que representa una reducción del error de más del 80%.Esto demuestra que XGBRegressor es mucho mejor que el DummyRegressor.La mejora indica que el modelo es capaz de aprender patrones complejos en los datos y hacer predicciones mucho más útiles y valiosas para la toma de decisiones empresariales de nuestra compañia de gaseosas.
Un MAE bajo, como el del modelo XGBoost, indica que el modelo predice con mayor precisión y puede ser utilizado como para Planificar inventario, optimizar logística y mejorar la rentabilidad y eficiencia.

In [20]:
import joblib

# Guardar modelo Dummy
joblib.dump(pipeline_dummy, 'modelo_dummy.pkl')

# Guardar modelo XGBoost
joblib.dump(pipeline_xgb, 'modelo_xgb.pkl')

print("Modelos guardados exitosamente como 'modelo_dummy.pkl' y 'modelo_xgb.pkl'")


Modelos guardados exitosamente como 'modelo_dummy.pkl' y 'modelo_xgb.pkl'


## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [22]:
# Inserte su código acá
from xgboost import XGBRegressor

# Creamos un preprocesador separado para acceder a las columnas ya transformadas
preprocessor_for_names = Pipeline([
    ('date', date_transformer),
    ('preprocess', preprocessor)
])
# Aplicamos preprocesamiento a X_train
X_train_transformed = preprocessor_for_names.fit_transform(X_train)
# Aplicamos primero la extracción de fechas por separado
X_train_date = date_transformer.fit_transform(X_train)

# Ahora aplicamos el ColumnTransformer directamente
X_train_transformed = preprocessor.fit_transform(X_train_date)

# Obtenemos los nombres de las columnas procesadas
feature_names = preprocessor.get_feature_names_out()

# Buscamos el índice del feature relacionado a 'price'
for i, name in enumerate(feature_names):
    if 'price' in name:
        print(f"Columna '{name}' en posición {i}")



Columna 'num__price' en posición 3


In [24]:
# Modelo XGBoost con restricción de monotonía en 'price'
# Ejemplo: solo el cuarto feature (índice 3) tiene restricción negativa
constraints = [0, 0, 0, -1] + [0] * (len(feature_names) - 4)

xgb_monotonic = XGBRegressor(
    random_state=42,
    monotone_constraints=tuple(constraints)
)

# Pipeline completo con restricciones
pipeline_monotonic = Pipeline(steps=[
    ('date', date_transformer),
    ('preprocess', preprocessor),
    ('regressor', xgb_monotonic)
])

# Entrenar
pipeline_monotonic.fit(X_train, y_train)

# Predecir y evaluar
y_pred_mono = pipeline_monotonic.predict(X_val)
mae_mono = mean_absolute_error(y_val, y_pred_mono)

print(f"MAE con restricción monótona en 'price': {mae_mono:.2f}")


MAE con restricción monótona en 'price': 2485.27


*Respuesta a pregunta 3*
El MAE cambia ligeramente a 2485, aunque el MAE subió un poco en comparación con el XGBRegressor sin restricciones, el modelo es más interpretable y coherente con la lógica del negocio, en teoría el amigo en esta ocasión NO tiene razon, ya que esto deberia haberse expresado en un baja del MAE, sin embargo cabe destacar que en general, si tiene razon por que la ley de demanda que nos dice anteniendo todo lo demás constante, cuando el precio sube, la cantidad demandada baja,habría que evaluar sobre una mayor cantidad de datos para ver si se presenta una baja del MAE, ya que en teoria esto si es cierto

In [27]:
# Guardar modelo XGBoost
joblib.dump(pipeline_monotonic ,'modelo_xgb_monotic.pkl')

print("Modelos guardados exitosamente como 'modelo_xgb_monotic.pkl")

Modelos guardados exitosamente como 'modelo_xgb_monotic.pkl


## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [28]:
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Inserte su código acá

import optuna
from optuna.samplers import TPESampler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Reusamos las funciones anteriores
def extract_date_parts(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%y')
    df['day'] = df['date'].dt.day.astype('category')
    df['month'] = df['date'].dt.month.astype('category')
    df['year'] = df['date'].dt.year.astype('category')
    return df.drop(columns=['date'])

date_transformer = FunctionTransformer(extract_date_parts)

# Features
numeric_features = ['lat', 'long', 'pop', 'price']
categorical_features = ['city', 'shop', 'brand', 'container', 'capacity', 'day', 'month', 'year']

# Datos
X_train_date = date_transformer.fit_transform(X_train)
X_val_date = date_transformer.transform(X_val)

# ✅ Objective Function para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    min_freq = trial.suggest_float("min_frequency", 0.0, 1.0)

    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 1.0)

    # OneHotEncoder personalizado
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=min_freq)

    # ColumnTransformer
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), numeric_features),
        ('cat', ohe, categorical_features)
    ]).set_output(transform='pandas')

    # Modelo XGBoost con restricción monótona en price (posición 3)
    constraints = [0, 0, 0, -1] + [0] * (len(numeric_features) - 4 + len(categorical_features))

    model = XGBRegressor(
        random_state=42,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        monotone_constraints=tuple(constraints)
    )

    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train_date, y_train)
    y_pred = pipeline.predict(X_val_date)
    mae = mean_absolute_error(y_val, y_pred)

    # Guardar el pipeline entrenado
    trial.set_user_attr("pipeline", pipeline)
    return mae


In [29]:
import time

study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=42)
)

# Optimizar durante 5 minutos
study.optimize(objective, timeout=300)  # 300 segundos = 5 minutos

# Reportar resultados
print(f"Número de trials: {len(study.trials)}")
print(f"Mejor MAE encontrado: {study.best_value:.2f}")
print("Mejores hiperparámetros encontrados:")
print(study.best_params)

# Obtener el mejor pipeline
mejor_pipeline = study.best_trial.user_attrs["pipeline"]


Número de trials: 159
Mejor MAE encontrado: 1914.67
Mejores hiperparámetros encontrados:
{'min_frequency': 0.05530999872353712, 'learning_rate': 0.08080550439574465, 'n_estimators': 780, 'max_depth': 8, 'max_leaves': 99, 'min_child_weight': 3, 'reg_alpha': 0.08147672000324523, 'reg_lambda': 0.9733056020350473}


*Respuesta a pregunta 3*
Con la optimización bayesiana realizada mediante **Optuna**, el MAE disminuyó a **1914**, mejorando significativamente respecto a los valores obtenidos previamente con `DummyRegressor` (13.298), `XGBRegressor` sin restricciones (2433) y `XGBRegressor` con restricción monótona (2485). Esta mejora se debe a que **Optuna ajustó automáticamente los hiperparámetros clave** del modelo y del `OneHotEncoder`, encontrando una configuración que mejor se adapta a los datos. Al buscar de forma inteligente en el espacio de parámetros usando TPE, se logró un modelo más preciso, con una mejor capacidad para capturar los patrones reales de la demanda. Esto demuestra el valor práctico de la optimización de hiperparámetros para mejorar el rendimiento en tareas predictivas.


*Respuesta a pregunta 4*
XGBRegressor es una implementación eficiente y optimizada del algoritmo de Gradient Boosting para problemas de regresión. El objetivo principal de este modelo es predecir un valor numérico continuo, como por ejemplo la cantidad de productos vendidos.

El modelo funciona construyendo una serie de árboles de decisión de forma secuencial, donde cada nuevo árbol se entrena para corregir los errores que cometieron los árboles anteriores

learning_rate
Este hiperparámetro controla cuánto se ajustan las predicciones en cada árbol. Si es muy alto, el modelo puede aprender demasiado rápido y sobreajustar. Si es muy bajo, el modelo aprenderá lentamente pero de forma más estable. Es una especie de velocidad de aprendizaje. El rango usado es el estándar para learning_rate en modelos de boosting. Valores más bajos como 0.001 hacen el aprendizaje más lento pero más preciso, mientras que valores como 0.1 permiten un entrenamiento más rápido con riesgo moderado de sobreajuste.

n_estimators
Es la cantidad total de árboles que se construyen durante el entrenamiento. Un mayor número de árboles puede permitir que el modelo aprenda mejor, pero también incrementa el tiempo de entrenamiento y el riesgo de sobreajuste si no se regula adecuadamente. Tambien hace sentido pues el rango permite explorar modelos desde rápidos (50 árboles) hasta modelos más precisos y complejos (1000 árboles).

max_depth
Define cuán profundo puede ser cada árbol individual. Un árbol más profundo puede aprender relaciones más complejas en los datos, pero también puede adaptarse demasiado a los datos de entrenamiento. Controlar la profundidad ayuda a evitar el sobreajuste. El rango controla la complejidad de cada árbol. Profundidades menores a 3 pueden ser demasiado simples y no capturar patrones importantes, mientras que profundidades mayores a 10 suelen generar sobreajuste.
max_leaves
Establece la cantidad máxima de hojas que puede tener cada árbol. Las hojas representan las predicciones finales del árbol. Limitar este número permite regular la complejidad del árbol sin depender solamente de la profundidad.Limitar el número de hojas permite controlar la complejidad del árbol más allá de su profundidad. Un rango de hasta 100 hojas es amplio y razonable para permitir variaciones de estructura sin llegar a árboles excesivamente complejos. El valor 0 permite que el modelo determine automáticamente cuántas hojas usar.

min_child_weight
Es la cantidad mínima de datos que debe tener una partición para que se le permita hacer una nueva división. Si este valor es alto, el modelo será más conservador al dividir los nodos. Esto ayuda a evitar que el modelo aprenda patrones ruidosos o irrelevantes.. El rango propuesto es adecuado para controlar esta regularización sin limitar excesivamente el modelo.

reg_alpha
Corresponde a la regularización L1. Penaliza las variables menos importantes y puede hacer que algunas se eliminen completamente. Esto ayuda a simplificar el modelo y reducir el riesgo de sobreajuste. El rango completo de 0 a 1 permite explorar desde ningún efecto de regularización hasta penalizaciones fuertes, lo que es útil para prevenir sobreajuste en datasets con muchas variables categóricas expandidas por one-hot encoding.

reg_lambda
Es la regularización L2. Penaliza los valores grandes en los parámetros del modelo de manera más suave que la regularización L1. Ayuda a estabilizar el modelo y mejorar su capacidad de generalización.La regularización L2 estabiliza el modelo ante datos ruidosos. El rango entre 0 y 1 es típico y suficiente para encontrar un buen balance entre underfitting y overfitting.

monotone_constraints
Este parámetro permite imponer relaciones lógicas entre ciertas variables y el valor a predecir. Por ejemplo, se puede indicar que a medida que sube el precio, la cantidad vendida debería disminuir. Esto mejora la interpretabilidad y coherencia del modelo con respecto a principios del negocio.

min_frequency (OneHotEncoder)
Este parámetro define qué tan frecuente debe ser una categoría para que se mantenga como una columna en la codificación one-hot. Las categorías con frecuencia menor a este valor se agrupan en una única categoría llamada "otros". Esto ayuda a reducir el número de columnas y a evitar que el modelo sobreaprenda categorías muy poco representadas.. Al permitir desde 0 (codifica todo) hasta 1 (codifica solo categorías dominantes), se exploran casos con alta granularidad y casos más compactos. Es útil cuando hay muchas categorías.

In [30]:
import joblib

# Guardar en archivo .pkl
joblib.dump(mejor_pipeline, 'modelo_xgb_optuna.pkl')

print("Modelo guardado exitosamente en 'modelo_xgb_optuna.pkl'")


Modelo guardado exitosamente en 'modelo_xgb_optuna.pkl'


## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [ ]:
#!pip install optuna-integration[xgboost]

In [ ]:
# Inserte su código acá

## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [ ]:
# Inserte su código acá

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [ ]:
# Inserte su código acá

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por mail o U-cursos.

<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=87110296-876e-426f-b91d-aaf681223468' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>